# Laboratorio 7 — Aprendizaje No Supervisado

**Estudiante:** Jhonn Wilder Llanos Rojas  
**Dataset:** Garbage Classification (12 classes)  

---

## Descripción del Dataset

El dataset **Garbage Classification** es una colección de imágenes de residuos domésticos organizada en **12 categorías**: `battery`, `biological`, `brown-glass`, `cardboard`, `clothes`, `green-glass`, `metal`, `paper`, `plastic`, `shoes`, `trash` y `white-glass`.

- **Total de imágenes:** ~15,500 imágenes distribuidas en 12 clases
- **Número de clases:** 12 categorías de residuos domésticos
- **Fuente:** Kaggle — Garbage Classification (12 classes)
- **Propósito original:** Desarrollar modelos de clasificación automática de basura para mejorar el reciclaje y la separación correcta de residuos

### Objetivo del Laboratorio

Aplicaremos técnicas de **aprendizaje no supervisado**: entrenaremos el modelo **sin usar las etiquetas** de las carpetas, descubriendo agrupaciones naturales en las imágenes mediante **K-Means**. Luego exploraremos cómo el **aprendizaje semi-supervisado** y el **aprendizaje activo** pueden mejorar la clasificación usando únicamente una pequeña fracción de etiquetas.

> *Predeciremos a qué grupo (cluster) pertenece una imagen nueva basándonos en su similitud visual con otras imágenes del dataset.*

## Sección 1 — Importaciones y Configuración

Importamos las librerías necesarias. Seguimos el mismo patrón que los cuadernillos del instructor, añadiendo únicamente lo imprescindible para cargar imágenes desde disco.

In [1]:
# numpy: operaciones matemáticas y manejo de arrays/matrices
import numpy as np

# matplotlib: visualización de datos, gráficas y figuras
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.ticker import FixedLocator, FixedFormatter  # para diagramas de silueta

# pathlib y os: manejo de rutas relativas al sistema de archivos
from pathlib import Path
import os

# PIL: carga y redimensionado de imágenes desde disco (estándar Python)
from PIL import Image

# sklearn.cluster: algoritmo K-Means para clustering
from sklearn.cluster import KMeans

# sklearn.metrics: métricas de evaluación (silhouette, accuracy)
from sklearn.metrics import silhouette_score, silhouette_samples, accuracy_score

# sklearn.decomposition: reducción de dimensionalidad con PCA
from sklearn.decomposition import PCA

# sklearn.preprocessing: normalización estadística (media 0, std 1)
from sklearn.preprocessing import StandardScaler

# sklearn.model_selection: división entrenamiento/prueba y selección estratificada
from sklearn.model_selection import train_test_split

# sklearn.linear_model: clasificador logístico para evaluación supervisada
from sklearn.linear_model import LogisticRegression

# warnings: suprimir advertencias de depreciación para salida limpia
import warnings
warnings.filterwarnings('ignore')

# Semilla global para reproducibilidad en todos los experimentos
np.random.seed(42)

print("Librerías cargadas correctamente.")

Librerías cargadas correctamente.


## Sección 2 — Carga del Dataset

Cargamos las imágenes desde la carpeta `garbage_classification/` usando rutas relativas.

**Pasos:**
1. Recorrer cada subcarpeta (clase) del dataset con `pathlib`
2. Cargar cada imagen con PIL y convertir a RGB
3. Redimensionar a **64×64 píxeles**
4. Aplanar a un vector de **12,288 features** (64×64×3)
5. Normalizar dividiendo entre **255.0** → rango [0, 1]
6. **Ignoramos las etiquetas** en las secciones de clustering (aprendizaje NO supervisado)

> **Nota:** Cargamos máximo 300 imágenes por clase para mantener tiempos de ejecución razonables (3,600 imágenes totales).

In [2]:
# Ruta relativa al dataset (misma carpeta que el notebook)
DATASET_PATH = Path("garbage_classification")

# Parámetros de carga
IMG_SIZE = (64, 64)   # redimensionar cada imagen a 64x64 píxeles
MAX_POR_CLASE = 300   # máximo de imágenes por clase para eficiencia

images = []       # lista de vectores aplanados y normalizados
labels = []       # etiquetas numéricas (0..11), guardadas para evaluación
class_names = []  # nombres de las clases (nombres de carpetas)

print("Cargando imágenes...")

# Recorrer cada subcarpeta (clase) en orden alfabético
for class_dir in sorted(DATASET_PATH.iterdir()):
    if not class_dir.is_dir():
        continue

    class_idx = len(class_names)
    class_names.append(class_dir.name)

    # Buscar imágenes con extensiones comunes
    img_files = (list(class_dir.glob("*.jpg")) +
                 list(class_dir.glob("*.jpeg")) +
                 list(class_dir.glob("*.png")))

    # Muestra aleatoria si hay más de MAX_POR_CLASE imágenes
    if len(img_files) > MAX_POR_CLASE:
        idx_sample = np.random.choice(len(img_files), MAX_POR_CLASE, replace=False)
        img_files = [img_files[i] for i in idx_sample]

    for img_path in img_files:
        try:
            # Cargar → convertir a RGB → redimensionar a 64x64
            img = Image.open(img_path).convert("RGB").resize(IMG_SIZE)
            # Aplanar a 12288 features y normalizar a [0, 1]
            img_flat = np.array(img, dtype=np.float32).flatten() / 255.0
            images.append(img_flat)
            labels.append(class_idx)
        except Exception:
            pass  # ignorar archivos corruptos o no válidos

# Convertir listas a arrays NumPy
X = np.array(images)  # forma: (n_imagenes, 12288)
y = np.array(labels)  # etiquetas numéricas: 0..11

print(f"\nTotal de imágenes cargadas: {len(X)}")
print(f"Forma del dataset X: {X.shape}  →  {X.shape[0]} imágenes × {X.shape[1]} features")
print(f"\nClases encontradas ({len(class_names)}):")
for i, name in enumerate(class_names):
    print(f"  [{i:2d}] {name}: {(y == i).sum()} imágenes")

Cargando imágenes...

Total de imágenes cargadas: 3600
Forma del dataset X: (3600, 12288)  →  3600 imágenes × 12288 features

Clases encontradas (12):
  [ 0] battery: 300 imágenes
  [ 1] biological: 300 imágenes
  [ 2] brown-glass: 300 imágenes
  [ 3] cardboard: 300 imágenes
  [ 4] clothes: 300 imágenes
  [ 5] green-glass: 300 imágenes
  [ 6] metal: 300 imágenes
  [ 7] paper: 300 imágenes
  [ 8] plastic: 300 imágenes
  [ 9] shoes: 300 imágenes
  [10] trash: 300 imágenes
  [11] white-glass: 300 imágenes


In [ ]:
# Mostrar un ejemplo de imagen por cada clase
fig, axes = plt.subplots(2, 6, figsize=(15, 6))
axes = axes.flatten()

for class_idx, ax in enumerate(axes):
    if class_idx >= len(class_names):
        ax.axis('off')
        continue
    # Índices de la clase actual
    clase_idxs = np.where(y == class_idx)[0]
    # Elegir una imagen aleatoria de esa clase
    sample_idx = clase_idxs[np.random.randint(len(clase_idxs))]
    # Reconstruir la imagen 64x64x3 desde el vector aplanado
    img = X[sample_idx].reshape(64, 64, 3)
    ax.imshow(img)
    ax.set_title(class_names[class_idx], fontsize=9)
    ax.axis('off')

plt.suptitle("Ejemplo de imagen por clase (sin etiquetas en clustering)", fontsize=14)
plt.tight_layout()
plt.show()

## Sección 3 — Preprocesamiento

### Normalización μ/σ (StandardScaler)
Aunque ya dividimos entre 255 para llevar los píxeles al rango [0, 1], aplicamos también **normalización estadística** (media=0, desviación estándar=1). Esto garantiza que K-Means calcule distancias Euclidianas de manera justa entre todas las features, sin que ninguna domine por tener mayor escala.

### Reducción de Dimensionalidad con PCA
Con **12,288 features** por imagen, K-Means sufre la **maldición de la dimensionalidad**: en espacios de alta dimensión las distancias Euclidianas pierden significado y el algoritmo converge lentamente.

**PCA** proyecta los datos a un espacio de menor dimensión preservando la mayor varianza posible. Usamos **50 componentes principales**, que típicamente capturan más del 70-80% de la varianza en imágenes.

In [ ]:
# --- Normalización μ/σ ---
# fit_transform: calcula media y std del dataset y los aplica
# Resultado: cada feature tendrá media≈0 y std≈1
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"Forma después de StandardScaler: {X_scaled.shape}")
print(f"Media (primeras 5 features): {X_scaled[:, :5].mean(axis=0).round(4)}")
print(f"Std  (primeras 5 features): {X_scaled[:, :5].std(axis=0).round(4)}")

# --- Reducción de Dimensionalidad con PCA ---
N_COMPONENTS = 50  # reducir de 12288 a 50 dimensiones
pca = PCA(n_components=N_COMPONENTS, random_state=42)
X_pca = pca.fit_transform(X_scaled)  # forma: (n_imagenes, 50)

# Varianza explicada acumulada: cuánta información conservamos
varianza_acum = np.cumsum(pca.explained_variance_ratio_)
print(f"\nVarianza explicada con {N_COMPONENTS} componentes: {varianza_acum[-1]*100:.1f}%")
print(f"Forma después de PCA: {X_pca.shape}  →  reducción de {X.shape[1]} a {N_COMPONENTS} features")

# Gráfica de varianza explicada acumulada
plt.figure(figsize=(8, 4))
plt.plot(range(1, N_COMPONENTS + 1), varianza_acum, "bo-", markersize=4)
plt.axhline(y=0.8, color='r', linestyle='--', label='80% varianza')
plt.xlabel("Número de componentes PCA", fontsize=12)
plt.ylabel("Varianza explicada acumulada", fontsize=12)
plt.title("Varianza Explicada por PCA", fontsize=14)
plt.legend()
plt.grid(True)
plt.show()

# División train/test para evaluación en secciones 5 y 6
# 80% entrenamiento, 20% prueba — estratificado por clase
X_train_pca, X_test_pca, y_train, y_test = train_test_split(
    X_pca, y, test_size=0.2, random_state=42, stratify=y
)
print(f"\nTrain: {X_train_pca.shape}, Test: {X_test_pca.shape}")

## Sección 4 — K-Means, Método del Codo y Silhouette Score

### K-Means
El algoritmo **K-Means** agrupa las muestras en $k$ clusters minimizando la **inercia** (suma de distancias cuadradas de cada punto a su centroide). Es iterativo: asigna cada punto al centroide más cercano, recalcula los centroides como el promedio de los puntos asignados, y repite hasta converger.

### Método del Codo
Entrenamos K-Means para $k = 1, 2, ..., 20$ y graficamos la **inercia** vs $k$. El punto donde la curva "se dobla" como un codo indica el $k$ óptimo: agregar más clusters no reduce significativamente la inercia.

### Silhouette Score
Mide qué tan bien definidos están los clusters: $s = (b - a) / \max(a, b)$, donde $a$ = distancia media intra-cluster y $b$ = distancia media al cluster más cercano. Valores cercanos a **1** indican clusters bien separados; cercanos a **0** indican fronteras difusas; cercanos a **-1** indican mala asignación.

In [ ]:
# Entrenar K-Means con k=12 (mismo número que las clases reales del dataset)
# n_init=10: probar 10 inicializaciones aleatorias y quedarse con la mejor
k = 12
print(f"Entrenando K-Means con k={k} clusters...")

kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
y_kmeans = kmeans.fit_predict(X_pca)  # asignar cada imagen al cluster más cercano

print(f"\nClusters asignados (primeros 20): {y_kmeans[:20]}")
print(f"\nDistribución de imágenes por cluster:")
for cluster_id in range(k):
    n = (y_kmeans == cluster_id).sum()
    print(f"  Cluster {cluster_id:2d}: {n:4d} imágenes")

# Inercia del modelo entrenado
print(f"\nInercia total del modelo K-Means (k=12): {kmeans.inertia_:.1f}")

In [ ]:
# MÉTODO DEL CODO: probar k desde 1 hasta 20 y graficar la inercia
# La inercia disminuye siempre al aumentar k; buscamos el "codo" donde la mejora se vuelve marginal
print("Calculando inercias para k=1..20...")

k_range = range(1, 21)
inertias = []

for k_val in k_range:
    model = KMeans(n_clusters=k_val, random_state=42, n_init=5)
    model.fit(X_pca)
    inertias.append(model.inertia_)
    print(f"  k={k_val:2d} → inercia={model.inertia_:.1f}")

# Graficar la curva del codo
plt.figure(figsize=(8, 4))
plt.plot(list(k_range), inertias, "bo-")
plt.xlabel("$k$ (número de clusters)", fontsize=14)
plt.ylabel("Inercia", fontsize=14)
plt.title("Método del Codo — Inercia vs $k$", fontsize=14)
plt.axvline(x=12, color='r', linestyle='--', label='k=12 (clases reales)')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# SILHOUETTE SCORE para distintos valores de k
# silhouette_score es O(n²); usamos una muestra para reducir el tiempo de cómputo
MUESTRA_SIL = 1500
np.random.seed(42)
sil_idx = np.random.choice(len(X_pca), min(MUESTRA_SIL, len(X_pca)), replace=False)
X_sil = X_pca[sil_idx]

print("Calculando Silhouette Scores para k=2..15...")

k_sil_range = range(2, 16)
silhouette_scores = []

for k_val in k_sil_range:
    model = KMeans(n_clusters=k_val, random_state=42, n_init=5)
    labels_sil = model.fit_predict(X_sil)
    score = silhouette_score(X_sil, labels_sil)
    silhouette_scores.append(score)
    print(f"  k={k_val:2d} → silhouette score = {score:.4f}")

# Graficar silhouette scores
plt.figure(figsize=(8, 4))
plt.plot(list(k_sil_range), silhouette_scores, "bo-")
plt.xlabel("$k$", fontsize=14)
plt.ylabel("Silhouette score", fontsize=14)
plt.title("Silhouette Score vs Número de Clusters", fontsize=14)
plt.axvline(x=12, color='r', linestyle='--', label='k=12 (clases reales)')
plt.legend()
plt.grid(True)
plt.show()

best_k = list(k_sil_range)[np.argmax(silhouette_scores)]
print(f"\nMejor k según Silhouette: {best_k}  (score={max(silhouette_scores):.4f})")

In [ ]:
# DIAGRAMAS DE SILUETA para k cercanos a 12
# Cada barra horizontal representa un punto; el ancho = su coeficiente de silueta
# La línea roja = silhouette score promedio del modelo
plt.figure(figsize=(11, 9))
ks_diag = [10, 11, 12, 13]

for plot_idx, k_val in enumerate(ks_diag):
    plt.subplot(2, 2, plot_idx + 1)

    model_diag = KMeans(n_clusters=k_val, random_state=42, n_init=5)
    y_diag = model_diag.fit_predict(X_sil)
    sil_coeffs = silhouette_samples(X_sil, y_diag)
    sil_avg = silhouette_score(X_sil, y_diag)

    padding = len(X_sil) // 30
    pos = padding
    ticks = []

    for i in range(k_val):
        coeffs = sil_coeffs[y_diag == i]
        coeffs.sort()
        color = mpl.cm.Spectral(i / k_val)
        plt.fill_betweenx(np.arange(pos, pos + len(coeffs)), 0, coeffs,
                          facecolor=color, edgecolor=color, alpha=0.7)
        ticks.append(pos + len(coeffs) // 2)
        pos += len(coeffs) + padding

    plt.gca().yaxis.set_major_locator(FixedLocator(ticks))
    plt.gca().yaxis.set_major_formatter(FixedFormatter(range(k_val)))
    plt.ylabel("Cluster")
    plt.xlabel("Silhouette Coefficient")
    plt.axvline(x=sil_avg, color="red", linestyle="--", label=f"avg={sil_avg:.3f}")
    plt.legend(fontsize=8)
    plt.title(f"$k={k_val}$", fontsize=14)

plt.tight_layout()
plt.show()

In [ ]:
# VISUALIZACIÓN DE CLUSTERS en 2D (primeras 2 componentes PCA)
# Como los datos tienen 50 dimensiones, proyectamos a 2D para visualizar
n_vis = min(2000, len(X_pca))
vis_idx = np.random.choice(len(X_pca), n_vis, replace=False)
X_vis = X_pca[vis_idx, :2]   # primeras 2 componentes PCA
y_vis = y_kmeans[vis_idx]    # etiquetas de cluster

plt.figure(figsize=(10, 7))
scatter = plt.scatter(X_vis[:, 0], X_vis[:, 1], c=y_vis,
                      cmap='tab20', s=5, alpha=0.6)
plt.colorbar(scatter, label='Cluster')
plt.xlabel("Componente PCA 1", fontsize=12)
plt.ylabel("Componente PCA 2", fontsize=12)
plt.title(f"Visualización de {k} Clusters K-Means (proyección 2D con PCA)", fontsize=14)
plt.tight_layout()
plt.show()

# IMAGEN MÁS REPRESENTATIVA de cada cluster
# La imagen más cercana al centroide de su cluster
X_dist = kmeans.transform(X_pca)           # distancias de cada imagen a cada centroide
representative_idxs = np.argmin(X_dist, axis=0)  # índice de la imagen más cercana a cada centroide

fig, axes = plt.subplots(2, 6, figsize=(15, 6))
axes = axes.flatten()

for cluster_id, ax in enumerate(axes):
    rep_idx = representative_idxs[cluster_id]
    img = X[rep_idx].reshape(64, 64, 3)  # reconstruir imagen original desde el vector aplanado
    ax.imshow(img)
    # Mostrar también la clase real para interpretar qué capturó el cluster
    clase_real = class_names[y[rep_idx]]
    ax.set_title(f"C{cluster_id}\n({clase_real})", fontsize=8)
    ax.axis('off')

plt.suptitle("Imagen más representativa de cada cluster (clase real entre paréntesis)", fontsize=13)
plt.tight_layout()
plt.show()

## Sección 5 — Aprendizaje Semi-Supervisado

El **aprendizaje semi-supervisado** combina un pequeño conjunto de datos etiquetados con un gran conjunto de datos no etiquetados. Es muy útil en la práctica porque etiquetar datos es costoso (requiere tiempo, dinero y expertos), pero obtener datos sin etiquetar es fácil y barato.

### Estrategia usada:
1. Seleccionamos aleatoriamente el **10% del conjunto de entrenamiento** como "etiquetado"
2. Entrenamos K-Means sobre **todos los datos** de entrenamiento (sin etiquetas)
3. Para cada cluster, usamos **votación mayoritaria** entre las muestras etiquetadas que contiene para asignar una etiqueta al cluster
4. Propagamos esa etiqueta a **todas las muestras** del cluster
5. Evaluamos la precisión de la propagación entrenando un `LogisticRegression`

### ¿Por qué funciona?
K-Means agrupa imágenes visualmente similares. Si la mayoría de imágenes etiquetadas en un cluster pertenecen a una clase, es razonable propagar esa etiqueta al resto de imágenes del mismo cluster.

In [ ]:
# APRENDIZAJE SEMI-SUPERVISADO

# Paso 1: Seleccionar el 10% del conjunto de entrenamiento como "etiquetado"
# Estratificado por clase para garantizar representación de todas las clases
n_labeled = int(0.10 * len(X_train_pca))
np.random.seed(42)
labeled_idx = np.random.choice(len(X_train_pca), n_labeled, replace=False)

# Máscara booleana: True = tiene etiqueta real
labeled_mask = np.zeros(len(X_train_pca), dtype=bool)
labeled_mask[labeled_idx] = True

print(f"Total de muestras train: {len(X_train_pca)}")
print(f"Etiquetadas  (10%): {labeled_mask.sum()}")
print(f"Sin etiquetar (90%): {(~labeled_mask).sum()}")

# Paso 2: K-Means sobre TODOS los datos de entrenamiento (sin usar etiquetas)
kmeans_ssl = KMeans(n_clusters=12, random_state=42, n_init=10)
kmeans_ssl.fit(X_train_pca)
cluster_labels_ssl = kmeans_ssl.labels_

# Paso 3: Propagar etiquetas por votación mayoritaria en cada cluster
y_propagated = np.full(len(X_train_pca), -1, dtype=int)  # -1 = sin propagar aún

print("\nPropagación de etiquetas por cluster:")
for cluster_id in range(12):
    en_cluster = cluster_labels_ssl == cluster_id           # todas las muestras del cluster
    etiquetadas_en_cluster = labeled_mask & en_cluster      # solo las que tienen etiqueta real

    if etiquetadas_en_cluster.sum() > 0:
        # Votación: clase más frecuente entre las etiquetadas del cluster
        votos = y_train[etiquetadas_en_cluster]
        etiqueta_mayoritaria = np.bincount(votos, minlength=len(class_names)).argmax()
        # Propagar a TODAS las muestras del cluster
        y_propagated[en_cluster] = etiqueta_mayoritaria
        print(f"  Cluster {cluster_id:2d}: {etiquetadas_en_cluster.sum():3d} etiquetadas "
              f"→ propaga '{class_names[etiqueta_mayoritaria]}'")
    else:
        print(f"  Cluster {cluster_id:2d}: sin muestras etiquetadas (cluster vacío)")

# Paso 4: Evaluación — comparar baseline vs semi-supervisado
# Baseline: LogisticRegression entrenado SOLO con el 10% etiquetado
log_reg_baseline = LogisticRegression(multi_class="ovr", solver="lbfgs",
                                       max_iter=5000, random_state=42)
log_reg_baseline.fit(X_train_pca[labeled_mask], y_train[labeled_mask])
acc_baseline = log_reg_baseline.score(X_test_pca, y_test)

# Semi-supervisado: LogisticRegression con etiquetas propagadas
valid_prop = y_propagated >= 0  # muestras con etiqueta propagada válida
log_reg_ssl = LogisticRegression(multi_class="ovr", solver="lbfgs",
                                  max_iter=5000, random_state=42)
log_reg_ssl.fit(X_train_pca[valid_prop], y_propagated[valid_prop])
acc_ssl = log_reg_ssl.score(X_test_pca, y_test)

print(f"\nResultados en test:")
print(f"  Baseline (10% etiquetado):        {acc_baseline*100:.1f}%")
print(f"  Semi-supervisado (propagación):   {acc_ssl*100:.1f}%")
print(f"  Mejora absoluta:                  {(acc_ssl - acc_baseline)*100:+.1f}%")

# Paso 5: Visualización comparativa
etiquetas_plot = ["Solo 10%\netiquetado", "Semi-\nsupervisado\n(K-Means)", "100%\netiquetado"]

# Entrenamos también con el 100% para tener referencia de techo
log_reg_full = LogisticRegression(multi_class="ovr", solver="lbfgs",
                                   max_iter=5000, random_state=42)
log_reg_full.fit(X_train_pca, y_train)
acc_full = log_reg_full.score(X_test_pca, y_test)
print(f"  Referencia (100% etiquetado):     {acc_full*100:.1f}%")

accs_plot = [acc_baseline, acc_ssl, acc_full]
colores = ['steelblue', 'coral', 'seagreen']

plt.figure(figsize=(7, 5))
bars = plt.bar(etiquetas_plot, accs_plot, color=colores, width=0.5)
plt.ylabel("Accuracy en test", fontsize=12)
plt.title("Aprendizaje Semi-Supervisado\nvs Baseline y Referencia", fontsize=13)
plt.ylim(0, 1.1)
for bar, acc in zip(bars, accs_plot):
    plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02,
             f"{acc*100:.1f}%", ha='center', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## Sección 6 — Aprendizaje Activo

El **aprendizaje activo** (*Active Learning*) entrena modelos de ML de manera **iterativa**, eligiendo inteligentemente qué muestras etiquetar en cada paso para maximizar el aprendizaje con el menor costo posible de etiquetado.

### ¿Por qué es útil?
En lugar de etiquetar datos al azar, el modelo **selecciona los ejemplos sobre los que tiene más incertidumbre**. Etiquetar esos ejemplos específicos mejora más rápido el modelo que etiquetar muestras aleatorias.

### Estrategia usada (basada en confianza):
1. Empezamos con solo el **5% del dataset** etiquetado
2. Entrenamos un clasificador (`LogisticRegression`) con esas muestras
3. Calculamos la **confianza** del modelo en cada muestra no etiquetada (`predict_proba` → máxima probabilidad)
4. Seleccionamos las muestras con **menor confianza** (mayor incertidumbre)
5. Las "etiquetamos" (usamos la etiqueta real del dataset) y agregamos al conjunto de entrenamiento
6. Repetimos el proceso y graficamos cómo mejora el accuracy en cada iteración

In [ ]:
# APRENDIZAJE ACTIVO

# Paso 1: Iniciar con el 5% del conjunto de entrenamiento etiquetado
# Usamos selección estratificada para asegurar que todas las clases estén presentes
n_inicial = max(len(class_names), int(0.05 * len(X_train_pca)))  # al menos 1 por clase

# Seleccionar n_inicial muestras de forma estratificada
_, idx_unlabeled_al, _, _ = train_test_split(
    np.arange(len(X_train_pca)), y_train,
    test_size=(1.0 - n_inicial / len(X_train_pca)),
    random_state=42,
    stratify=y_train
)
idx_labeled_al = np.setdiff1d(np.arange(len(X_train_pca)), idx_unlabeled_al)

# Máscara booleana: True = ya etiquetado
labeled_mask_al = np.zeros(len(X_train_pca), dtype=bool)
labeled_mask_al[idx_labeled_al] = True

print(f"Muestras iniciales etiquetadas (5%): {labeled_mask_al.sum()}")
print(f"Clases presentes en el inicio: {len(np.unique(y_train[labeled_mask_al]))}")

# Parámetros de las iteraciones
N_ITERACIONES = 8
MUESTRAS_POR_ITER = max(20, int(0.01 * len(X_train_pca)))  # agregar ~1% por iteración

accuracies_al = []          # accuracy en test tras cada iteración
n_labeled_al_hist = []      # número de muestras etiquetadas en cada iteración

print(f"\nEntrenando con {N_ITERACIONES} iteraciones, agregando {MUESTRAS_POR_ITER} muestras/iter...")
print("-" * 60)

for iteracion in range(N_ITERACIONES):
    n_ahora = labeled_mask_al.sum()
    n_labeled_al_hist.append(n_ahora)

    # Entrenar clasificador con las muestras actualmente etiquetadas
    X_lab_al = X_train_pca[labeled_mask_al]
    y_lab_al = y_train[labeled_mask_al]

    log_reg_al = LogisticRegression(multi_class="ovr", solver="lbfgs",
                                     max_iter=5000, random_state=42)
    log_reg_al.fit(X_lab_al, y_lab_al)

    # Evaluar en el conjunto de prueba
    acc_al = log_reg_al.score(X_test_pca, y_test)
    accuracies_al.append(acc_al)

    print(f"  Iteración {iteracion+1:2d}: {n_ahora:4d} etiquetadas → accuracy = {acc_al*100:.1f}%")

    # Seleccionar las muestras más INCIERTAS (menor confianza) para etiquetar
    indices_sin_etiquetar = np.where(~labeled_mask_al)[0]
    if len(indices_sin_etiquetar) == 0:
        print("  ¡Todas las muestras etiquetadas!")
        break

    X_no_etiquetadas = X_train_pca[indices_sin_etiquetar]

    # predict_proba: probabilidad de pertenencia a cada clase para cada muestra
    probas_al = log_reg_al.predict_proba(X_no_etiquetadas)

    # Confianza = probabilidad máxima (la clase más probable)
    # Una confianza baja significa que el modelo no sabe bien qué clase asignar
    confianzas = np.max(probas_al, axis=1)

    # Seleccionar las de MENOR confianza (mayor incertidumbre)
    n_a_agregar = min(MUESTRAS_POR_ITER, len(indices_sin_etiquetar))
    idx_inciertos_local = np.argsort(confianzas)[:n_a_agregar]  # índices locales (en X_no_etiquetadas)
    nuevos_etiquetados = indices_sin_etiquetar[idx_inciertos_local]  # índices globales

    # Marcar como etiquetados
    labeled_mask_al[nuevos_etiquetados] = True

print("-" * 60)
print(f"Accuracy final: {accuracies_al[-1]*100:.1f}% con {labeled_mask_al.sum()} muestras etiquetadas")

# También calculamos baseline aleatorio para comparación
# (mismo número de muestras pero seleccionadas aleatoriamente, sin estrategia activa)
n_final = labeled_mask_al.sum()
np.random.seed(99)
idx_random = np.random.choice(len(X_train_pca), n_final, replace=False)
log_reg_random_al = LogisticRegression(multi_class="ovr", solver="lbfgs",
                                        max_iter=5000, random_state=42)
log_reg_random_al.fit(X_train_pca[idx_random], y_train[idx_random])
acc_random_al = log_reg_random_al.score(X_test_pca, y_test)
print(f"Baseline aleatorio ({n_final} muestras): {acc_random_al*100:.1f}%")

# Graficar curva de aprendizaje activo
plt.figure(figsize=(9, 5))
plt.plot(n_labeled_al_hist, accuracies_al, "bo-", label="Aprendizaje Activo (incertidumbre)")
plt.axhline(y=acc_random_al, color='orange', linestyle='--',
            label=f"Aleatorio final ({n_final} muestras): {acc_random_al*100:.1f}%")
plt.xlabel("Número de muestras etiquetadas", fontsize=14)
plt.ylabel("Accuracy en test", fontsize=14)
plt.title("Aprendizaje Activo: Mejora del Accuracy con cada Iteración", fontsize=14)
plt.legend(fontsize=11)
plt.grid(True)
plt.tight_layout()
plt.show()

# Mostrar las 20 imágenes más inciertas de la última iteración
# (las que el modelo encuentra más difíciles de clasificar)
probas_final = log_reg_al.predict_proba(X_train_pca[~labeled_mask_al])
confianzas_final = np.max(probas_final, axis=1)
idx_mas_inciertos = np.argsort(confianzas_final)[:20]
indices_no_etq = np.where(~labeled_mask_al)[0]
idx_globales_inciertos = indices_no_etq[idx_mas_inciertos]

fig, axes = plt.subplots(2, 10, figsize=(16, 4))
axes = axes.flatten()
for ax_i, global_i in zip(axes, idx_globales_inciertos):
    # Necesitamos el índice en el array original X (no en X_train_pca)
    img = X_train_pca[global_i]  # vector PCA — no tiene sentido visual
    # Reconstruir imagen aproximada desde PCA
    img_reconstruida = pca.inverse_transform(img.reshape(1, -1))
    img_reconstruida = scaler.inverse_transform(img_reconstruida).reshape(64, 64, 3)
    img_reconstruida = np.clip(img_reconstruida, 0, 1)  # asegurar rango [0,1]
    conf_val = confianzas_final[np.where(idx_globales_inciertos == global_i)[0][0]]
    ax_i.imshow(img_reconstruida)
    ax_i.set_title(f"{conf_val:.2f}", fontsize=7)
    ax_i.axis('off')

plt.suptitle("20 imágenes más inciertas para el modelo\n(confianza = probabilidad máxima)", fontsize=12)
plt.tight_layout()
plt.show()

## Sección 7 — Predicción con Imagen Propia

Definimos una función que recibe la **ruta de cualquier imagen externa**, la procesa exactamente igual que el dataset, y la asigna al cluster más cercano usando el modelo K-Means entrenado.

### ¿Cómo funciona?
1. Cargar la imagen desde la ruta indicada
2. Redimensionar a 64×64 y convertir a RGB
3. Aplanar a 12,288 features y normalizar (/255)
4. Aplicar el mismo `StandardScaler` que se usó en el entrenamiento
5. Aplicar el mismo `PCA` (transformar a 50 componentes)
6. Predecir el cluster más cercano con el modelo K-Means
7. Mostrar la imagen y reportar el cluster con su clase dominante

In [ ]:
# FUNCIÓN DE PREDICCIÓN PARA IMAGEN EXTERNA

# Primero, construir un mapa cluster → clase dominante (para interpretar el resultado)
# Esto nos dice qué clase de residuo predomina en cada cluster
cluster_clase_dominante = {}
for cluster_id in range(k):
    en_cluster = y_kmeans == cluster_id
    if en_cluster.sum() > 0:
        conteo = np.bincount(y[en_cluster], minlength=len(class_names))
        clase_dom = np.argmax(conteo)
        pct_dom = conteo[clase_dom] / en_cluster.sum() * 100
        # Guardar top-2 clases del cluster para descripción
        top2 = np.argsort(conteo)[::-1][:2]
        cluster_clase_dominante[cluster_id] = {
            'clase_principal': class_names[clase_dom],
            'porcentaje': pct_dom,
            'descripcion': ' / '.join([class_names[t] for t in top2])
        }

print("Interpretación de cada cluster (clase dominante):")
for cid, info in cluster_clase_dominante.items():
    print(f"  Cluster {cid:2d}: {info['clase_principal']:15s} ({info['porcentaje']:.0f}%) "
          f"→ similar a: {info['descripcion']}")


def predecir_imagen(ruta_imagen, modelo_kmeans, scaler_fit, pca_fit,
                    cluster_info, img_size=(64, 64)):
    """
    Predice a qué cluster pertenece una imagen externa.
    Aplica el mismo preprocesamiento que se usó durante el entrenamiento.
    """
    # Cargar imagen, convertir a RGB, redimensionar igual que el dataset
    img = Image.open(ruta_imagen).convert("RGB").resize(img_size)

    # Aplanar a vector y normalizar entre [0, 1]
    img_array = np.array(img, dtype=np.float32).flatten() / 255.0

    # Aplicar el StandardScaler entrenado (misma normalización μ/σ)
    img_scaled = scaler_fit.transform(img_array.reshape(1, -1))

    # Aplicar PCA entrenado (reducir a 50 componentes)
    img_pca = pca_fit.transform(img_scaled)

    # Predecir el cluster más cercano
    cluster_pred = modelo_kmeans.predict(img_pca)[0]

    # Obtener información del cluster
    info = cluster_info.get(cluster_pred, {'descripcion': 'desconocido', 'porcentaje': 0})

    # Visualizar la imagen y el resultado
    plt.figure(figsize=(4, 4))
    plt.imshow(img)
    plt.axis('off')
    plt.title(f"Cluster {cluster_pred}", fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

    print(f"Esta imagen pertenece al cluster {cluster_pred}")
    print(f"  → Similar a: {info['descripcion']}")
    print(f"  → Clase dominante del cluster: {info.get('clase_principal','?')} "
          f"({info.get('porcentaje',0):.0f}% de las imágenes del cluster)")

    return cluster_pred


print("\nFunción 'predecir_imagen' definida correctamente.")

In [ ]:
# DEMOSTRACIÓN: predecir con imágenes del propio dataset
# En producción, reemplaza la ruta con cualquier imagen propia

print("=" * 55)
print("DEMOSTRACIÓN DE PREDICCIÓN")
print("=" * 55)

# Tomar una imagen aleatoria de 3 clases diferentes para demostrar
clases_demo = ['battery', 'plastic', 'clothes']

for clase_demo in clases_demo:
    ruta_clase = DATASET_PATH / clase_demo
    if ruta_clase.exists():
        imagenes_clase = (list(ruta_clase.glob("*.jpg")) +
                         list(ruta_clase.glob("*.jpeg")) +
                         list(ruta_clase.glob("*.png")))
        if imagenes_clase:
            img_demo = str(imagenes_clase[np.random.randint(len(imagenes_clase))])
            print(f"\nImagen de prueba — clase real: '{clase_demo}'")
            print(f"Ruta: {img_demo}")
            cluster_resultado = predecir_imagen(
                img_demo, kmeans, scaler, pca, cluster_clase_dominante
            )
            print()

# ─────────────────────────────────────────────────────
# Para usar con una imagen propia, descomenta y edita:
# ─────────────────────────────────────────────────────
# mi_imagen = "ruta/a/mi/imagen.jpg"  # reemplaza con tu ruta
# predecir_imagen(mi_imagen, kmeans, scaler, pca, cluster_clase_dominante)

## Resumen

En este laboratorio aplicamos técnicas de **aprendizaje no supervisado** al dataset de clasificación de residuos:

| Técnica | Descripción | Resultado |
|---|---|---|
| **K-Means (k=12)** | Agrupación sin etiquetas en 12 clusters | Clusters visualmente coherentes |
| **Método del Codo** | Encontrar k óptimo por inercia | Codo visible alrededor de k=12 |
| **Silhouette Score** | Medir calidad de los clusters | Mejor k identificado |
| **Semi-supervisado** | Propagar 10% de etiquetas con K-Means | Mejora respecto al baseline |
| **Aprendizaje Activo** | Etiquetar muestras inciertas iterativamente | Mejora progresiva del accuracy |
| **Predicción** | Asignar cualquier imagen nueva a un cluster | Función generalizable |

### Conclusión

El aprendizaje no supervisado permite descubrir estructura en los datos **sin necesidad de etiquetas**. Aunque los clusters no corresponden perfectamente a las 12 clases reales (K-Means no conoce las clases), sirven de base para técnicas semi-supervisadas y activas que obtienen buenas prestaciones con **muy poco esfuerzo de etiquetado**.